In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

spark = SparkSession.builder \
.config("spark.sql.warehouse/dir", f"/user/{username}/warehouse") \
.config("spark.ui.port", '0') \
.enableHiveSupport() \
.master("yarn") \
.appName("0253") \
.getOrCreate()

In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
product_schema = StructType([
    StructField("ProductID", IntegerType()),
    StructField("Category", StringType()),
    StructField("ProductName", StringType()),
    StructField("Description", StringType()),
    StructField("Price", FloatType()),
    StructField("ImageURL", StringType())
])
product_df = spark.read.format("csv").option("header", False).schema(product_schema).load("datasets/retail_db_product_part-00000")

### 4. Find the number of products in each category that have a price greater than $100. Display the results in a tabular format that shows the category name and the number of products that satisfy the condition.

In [7]:
product_df.filter("price > 100").groupBy("category").count()

category,count
7,6
51,7
54,6
11,19
29,9
42,4
3,5
30,17
34,15
8,5


In [3]:
product_df.createOrReplaceTempView("product")

In [6]:
spark.sql("select category, count(*) as number_of_products from product where Price > 100 group by category")

category,number_of_products
7,6
51,7
54,6
11,19
29,9
42,4
3,5
30,17
34,15
8,5
